<a href="https://colab.research.google.com/github/vedadapa/rag-pipeline/blob/main/sentence_transformers_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sentence_transformers import SentenceTransformer,util
# The "all-MiniLM-L6-v2" model is used to encode the sentences not the words in 384 dimensions
model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "Customer defaulted on home loan",
    "Borrower failed to repay mortgage",
    "The weather in Mumbai is hot today"
]
# Each sentence is allocated in specified variables as vector dimensions(eg. emb1)
emb1,emb2,emb3 = model.encode(sentences)
#comparing the the sentences to find the cosine similarity
sen1_2 = util.cos_sim(emb1,emb2)
sen1_3 = util.cos_sim(emb1,emb3)
print("sentence 1 and 2:",sen1_2)
print("sentence 1 and 3:",sen1_3)







In [ ]:
# each sentence is stored in 384 dimension space
print(emb1.shape)

(384,)


In [ ]:
# chromadb is a vector database to store and query embeddings.
import chromadb

# Initialize an in-memory ChromaDB client.
client = chromadb.Client()

# Get or create a collection for loan documents.
collection = client.get_or_create_collection("loan_docs")

# Define documents to be stored.
documents = [
    "Customer defaulted on home loan",
    "Borrower failed to repay mortgage",
    "The weather in Mumbai is hot today"
]

# Add documents to the collection with unique IDs.
collection.add(
    documents = documents,
    ids = ['doc1','doc2','doc3'])

# Query the collection for similar documents.
results = collection.query(
    query_texts= ["person didn't pay back their loan"],n_results=2
)

# Print the query results.
print(results)

{'ids': [['doc2', 'doc1']], 'embeddings': None, 'documents': [['Borrower failed to repay mortgage', 'Customer defaulted on home loan']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[0.70534747838974, 0.9512903690338135]]}


In [ ]:
from groq import Groq
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Initialize the Groq client with your API key from environment variables.
client = Groq(api_key = os.getenv("GROQ_API_KEY"))

# Define the context and question for the Groq API call.
context = "Borrower failed to repay mortgage. Customer defaulted on home loan."
question = "What happened with the loan?"

# Format the prompt as required by the model.
prompt = f"Answer the question based on this context only: {context}\n\nQuestion: {question}"

# Call the Groq API to get a chat completion.
response = client.chat.completions.create(
    model = "llama-3.1-8b-instant", # Specify the model to use.
    messages=[
    {"role": "user", "content": prompt} # Provide the user's message with the formatted prompt.
]

)
# Print the content of the response from the AI model.
print(response.choices[0].message.content)


The borrower failed to repay the mortgage, and the customer defaulted on their home loan.
